In [1]:
import pandas as pd
import numpy as np

import xarray as xr
import geopandas as gpd
import rioxarray as rxr
from shapely.geometry import mapping
import matplotlib.pyplot as plt

In [12]:
class DataLoader:
    ''' Class for loading climate data from differen sources and bring them in the desired format.
    For adding new methods: The result should be a xarray timeseries for your area of interest,
    for one scenario/model/variable combination.'''

    def crop_to_park_boundary(self, nc, boundary, variable):
        ''' Takes an xarray object and crops within the provided boundary (geopandas), returning a clipped xarray object. '''
        # Select just the variable (time_bnds doesn't have spatial dims)
        nc = nc[variable].rio.set_spatial_dims(x_dim="lon", y_dim="lat")
        nc = nc.rio.write_crs("EPSG:4326", inplace=True)
        if nc.rio.crs != boundary.crs:
            boundary = boundary.to_crs(nc.rio.crs)
        nc_clipped = nc.rio.clip(boundary.geometry.values, boundary.crs, drop=True, all_touched=True)

        return(nc_clipped)

    def load_isimip(self, scenario, model, variable, boundary):
        ''' Loads data from ISIMIP 3b. This data has alsoready been downloaded, and cropped to the area of interest,
        using bash and CDO. The scripts are in this repository. The data is founds in isimip/data/processed'''
        nc = xr.open_dataset(f'isimip/data/processed/{model}_w5e5_{scenario}_{variable}_jotr_monthly.nc', engine="netcdf4")
        nc_clipped = self.crop_to_park_boundary(nc, boundary, variable).mean(("lon", "lat"))

        return nc_clipped


In [ ]:
class ClimateFutures:
    ''' This class contains all the functions to create climate future datasets'''

    def __init__(self):
        self.loader = DataLoader()

    def calculate_anomaly(self, scenario, baseline_period, model, variable, boundary):
        ''' Takes a clipped xarray object and calculates the anomaly relative to the provided baseline period. 
        Returns an xarray object of the anomaly. '''

        load = DataLoader()

        nc = self.loader.load_isimip(scenario, model, variable, boundary)
        nc_hist = self.loader.load_isimip("historical", model, variable, boundary).mean(("lon", "lat"))
        baseline = nc_hist.sel(time=slice(baseline_period[0], baseline_period[1])).mean("time")
        anomaly = nc - baseline

        return(anomaly)
        